# 13 — End-to-end baselineの固定

改造前の設定と評価項目を固定し、比較可能な実験単位を作ります。

**前提**: `12_force_to_joint_torque.ipynb`

> 読み方: 「直感 → 数式 → 上流コード → 小実験 → 解釈」の順です。
> `実装事実` と書いた箇所は現行 `external/Quadruped-PyMPC` のコード、
> `学習用モデル` は理解のために単純化した再実装です。

In [1]:
# 背景: End-to-end baselineを上流PyMPCと同じリポジトリ配置・環境変数で固定します。
# 目的: ワークスペースとPyMPCの場所を確定し、後続セルの比較条件を再現可能にします。
# OSに依存しないパス演算を行うためPathを読み込む。
from pathlib import Path
# 環境変数の設定にos、モジュール検索パスの設定にsysを使う。
import os, sys

# Notebookを起動した現在位置を絶対パスへ正規化する。
ROOT = Path.cwd().resolve()
# notebook_pympc直下から起動した場合だけリポジトリルートへ移る。
if ROOT.name == "notebook_pympc":
    # 外部実装をROOT基準で参照できるよう親ディレクトリを採用する。
    ROOT = ROOT.parent
# 上流Quadruped-PyMPCの配置先をROOTから組み立てる。
PYMPC_ROOT = ROOT / "external" / "Quadruped-PyMPC"
# 誤った起動場所のままbaselineを記録しないよう実装の存在を検証する。
assert PYMPC_ROOT.exists(), f"Quadruped-PyMPC が見つかりません: {PYMPC_ROOT}"
# 同名モジュールの取り違えを防ぐため検索パス未登録時だけ処理する。
if str(PYMPC_ROOT) not in sys.path:
    # 現行リポジトリの実装を最優先でimportするため先頭へ追加する。
    sys.path.insert(0, str(PYMPC_ROOT))

# acados生成物の探索基準を未設定時だけ上流同梱ディレクトリへ合わせる。
os.environ.setdefault("ACADOS_SOURCE_DIR", str(PYMPC_ROOT / "quadruped_pympc" / "acados"))
# 画面のないbaseline試験でもMuJoCoを描画できるよう未設定時はEGLを選ぶ。
os.environ.setdefault("MUJOCO_GL", "egl")
# 実験が参照するワークスペースを目視確認できるよう表示する。
print("workspace :", ROOT)
# 上流実装の参照先を目視確認できるよう表示する。
print("PyMPC root:", PYMPC_ROOT)

workspace : /home/takuya/work/mpc_dog
PyMPC root: /home/takuya/work/mpc_dog/external/Quadruped-PyMPC


## Baseline契約

比較ごとに最低限、commit/tree、robot、scene、seed、速度指令、摩擦、
gait、MPC type、N、dt、Q/R、solver statusを保存します。
`config.py` のdisk既定sceneは `perlin` なので、平地試験は `flat` を明示します。

In [2]:
# 背景: Baseline比較にはrobot・scene・seed・指令・摩擦・gait・MPC離散化を一緒に固定する必要があります。
# 目的: 現行configと明示的な試験条件から再実行可能なbaseline契約をJSON形式で表示します。
# 上流実装が実際に使う設定値を参照するためconfigをcfgとして読み込む。
from quadruped_pympc import config as cfg
# 条件辞書を人が読めるJSONへ直列化するためjsonを使う。
import json
# 1試験を再現するために必要な条件をキー付き辞書へまとめる。
record = {
    # 使用するロボット機種を上流configから記録する。
    "robot": cfg.robot,
    # config既定値に依存せず比較用地形を平地flatへ固定する。
    "scene_for_experiment": "flat",
    # 確率的要素を再現できるよう乱数seedを0へ固定する。
    "seed": 0,
    # base座標の速度指令[vx,vy,yaw_rate]を[m/s,m/s,rad/s]で記録する。
    "command_mps": [0.2, 0.0, 0.0],
    # MuJoCo Plant側の地面摩擦係数を0.8として記録する。
    "ground_mu": 0.8,
    # MPC摩擦角錐が使う無次元係数μを上流configから記録する。
    "mpc_mu": cfg.mpc_params["mu"],
    # 接触scheduleを決める歩容名を上流configから記録する。
    "gait": cfg.simulation_params["gait"],
    # trotのstep周波数[Hz]を上流configから記録する。
    "step_freq": cfg.simulation_params["gait_params"]["trot"]["step_freq"],
    # 1周期中の立脚割合を表す無次元duty factorを記録する。
    "duty_factor": cfg.simulation_params["gait_params"]["trot"]["duty_factor"],
    # MPCの予測段数Nを上流configから記録する。
    "horizon": cfg.mpc_params["horizon"],
    # MPC各stageの離散時間dt[s]を上流configから記録する。
    "mpc_dt": cfg.mpc_params["dt"],
}
# baseline契約をインデント2のJSONとして表示し、比較時に保存できる形へする。
print(json.dumps(record, indent=2))

{
  "robot": "go2",
  "scene_for_experiment": "flat",
  "seed": 0,
  "command_mps": [
    0.2,
    0.0,
    0.0
  ],
  "ground_mu": 0.8,
  "mpc_mu": 0.42,
  "gait": "trot",
  "step_freq": 1.35,
  "duty_factor": 0.74,
  "horizon": 12,
  "mpc_dt": 0.02
}


## 重い実行

次のセルは意図的に既定OFFです。全身NMPCはsolver生成とMuJoCo計算を伴います。
実行するときだけ `RUN_HEAVY=True` にし、Jupyterを
`.env.workshop` source後に起動してください。

In [3]:
# 背景: 全身NMPC baselineはsolver生成とMuJoCo計算を伴うため、通常の教材実行では明示的に無効化します。
# 目的: opt-in時だけ固定条件の2 s headless試験を行い、終了時にはscene設定を必ず復元します。
# 重い閉ループ試験を誤って開始しないよう既定値をFalseにする。
RUN_HEAVY = False
# 利用者がTrueへ変更した場合だけ全身NMPCとMuJoCoを実行する。
if RUN_HEAVY:
    # 実行時点の上流設定を参照・一時変更するためconfigを読み込む。
    from quadruped_pympc import config as cfg
    # 上流と同じ閉ループsimulation entry pointを読み込む。
    from simulation.simulation import run_simulation
    # 試験後に副作用なく戻せるよう現在のscene名を保存する。
    old_scene = cfg.simulation_params["scene"]
    # baseline契約に従いPlant地形を明示的にflatへ変更する。
    cfg.simulation_params["scene"] = "flat"
    # simulationが失敗してもsceneを復元できるよう保護区間を開始する。
    try:
        # 上流の全身NMPC閉ループを固定条件で1 episode実行する。
        run_simulation(
            # 上流configをsimulation全体の設定入力として渡す。
            cfg,
            # 統計を混ぜない単一episodeを2 sだけ実行する。
            num_episodes=1, num_seconds_per_episode=2,
            # base座標の前進速度指令範囲を0.2〜0.2 m/sへ固定し、角速度を0 rad/sにする。
            ref_base_lin_vel=(0.2, 0.2), ref_base_ang_vel=(0., 0.),
            # Plant摩擦係数を0.8へ固定し、指令形式を前進のみにする。
            friction_coeff=(0.8, 0.8), base_vel_command_type="forward",
            # 確率的要素をseed=0で再現し、画面なしで実行する。
            seed=0, render=False,
        )
    # 正常終了・例外のどちらでも一時変更したsceneを戻す。
    finally:
        # Notebook後続実験へflat設定を持ち越さないよう元のsceneを復元する。
        cfg.simulation_params["scene"] = old_scene
# 既定の軽量経路では重い計算を行わず、明示的な案内だけ表示する。
else:
    # 実行を希望する場合の切替方法と試験時間を表示する。
    print("Skipped. Set RUN_HEAVY=True for a 2 s headless baseline.")

Skipped. Set RUN_HEAVY=True for a 2 s headless baseline.


成功判定は転倒しないことだけでは不十分です。速度RMSE、姿勢RMS/最大値、
高さ誤差、摩擦margin、トルク飽和率、solver失敗率、計算時間を同じ時間窓で比較します。

## 章末チェック

出力を眺めるだけでなく、次を自分の言葉で答えてください。

1. この章の入力・出力の shape、単位、座標系は何か。
2. 変更可能な量と、他の章から渡される量は何か。
3. パラメータを2倍にしたとき、どのグラフがどちらへ変化するか。
4. 現行実装の事実と、学習用の近似を区別できるか。